# Challenge 2: Workflow Graphs & Conditional Routing

## From Manual Chains to DAG Orchestration

In Challenge 1, you manually chained agents: triage → diagnostics → plan → verify.
That works for one path, but production incidents need **conditional routing**:

```
                          ┌─ [CRITICAL/HIGH] ─→ Diagnostics ─→ Comms
                          │
  Alert ─→ Triage ─→ Switch 
                          │
                          └─ [LOW] ─────────→ Monitor Only
```

MAF's `WorkflowBuilder` models this as a **directed graph** with:
- **Executors**: Nodes that process data (agents, transformers, custom logic)
- **Edges**: Connections between nodes, optionally with **conditions**
- **Switch-Case Edge Groups**: Deterministic one-of-N routing
- **State**: Shared data accessible by all executors via `ctx.set_state()`

---

## What You'll Build

| Component | MAF Concept | Purpose |
|-----------|-------------|--------|
| Ingest alert | `@executor` | Parse raw JSON, store in state, forward to agent |
| Triage agent | `AgentExecutor` | Wraps your triage Agent as a workflow node |
| Parse triage | `@executor` | Parses structured output, emits routing decision |
| Switch-case routing | `add_switch_case_edge_group` | Routes CRITICAL/HIGH vs LOW to different paths |
| To-diagnostics | `@executor` | Bridges routing to diagnostics agent |
| Diagnostics agent | `AgentExecutor` | Wraps your diagnostics Agent |
| Monitor-only | `@executor` | Terminal node for low-severity alerts |
| Communications | `@executor` | Terminal node that yields final report |

## Setup

In [1]:
import os
import sys
import json
from typing import Any, Literal
from dataclasses import dataclass

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Case,
    Default,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("✅ Imports ready — WorkflowBuilder, AgentExecutor, Case, Default loaded")

c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Imports ready — WorkflowBuilder, AgentExecutor, Case, Default loaded


---
## Step 1: Pydantic Models (from Challenge 1)

We reuse the structured output models. Copy your definitions from Challenge 1
or use these minimal versions:

In [2]:
class TriageResult(BaseModel):
    severity: Literal["critical", "high", "medium", "low"]
    is_recurring: bool
    auto_remediation_allowed: bool
    root_cause_hypothesis: str
    recommended_action: str
    escalation_threshold_minutes: int

class DiagnosticsResult(BaseModel):
    root_cause: str
    evidence: list[str]
    affected_components: list[str]
    confidence: float
    recommended_fix: str
    requires_restart: bool

print("✅ Models defined")

✅ Models defined


---
## Step 2: Create Agents

Same agents as Challenge 1. In the workflow, they'll be wrapped by `AgentExecutor`.

In [3]:
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

triage_agent = Agent(
    client,
    id="triage-agent",
    name="TriageAgent",
    instructions=(
        "You are an incident Triage Agent. When given an alert:\n"
        "1. Call check_alert_history with the service name\n"
        "2. Call get_runbook with the incident_type\n"
        "Then classify severity and recommend next steps."
    ),
    tools=[check_alert_history, get_runbook],
    default_options=OpenAIChatOptions(response_format=TriageResult),
)

diagnostics_agent = Agent(
    client,
    id="diagnostics-agent",
    name="DiagnosticsAgent",
    instructions=(
        "You are an incident Diagnostics Agent. Investigate the root cause:\n"
        "1. Call get_metrics for relevant metric types (memory, latency, cpu, error_rate)\n"
        "2. Call get_logs with severity='error'\n"
        "3. Call check_dependencies to identify cascading failures\n"
        "Provide confidence based on evidence quality."
    ),
    tools=[get_metrics, get_logs, check_dependencies],
    default_options=OpenAIChatOptions(response_format=DiagnosticsResult),
)

print("✅ Agents created")

✅ Agents created


---
## Step 3: Build Workflow Executors

Executors are the **nodes** in your workflow graph. Each receives typed input
and either sends a message downstream or yields workflow output.

### Key Concepts:
- `@executor(id="...")` — decorator to create a workflow node from a function
- `ctx.send_message(data)` — pass data to the next node (ASYNC — needs `await`)
- `ctx.yield_output(data)` — emit final workflow result (ASYNC — needs `await`)
- `ctx.set_state(key, value)` — store data accessible by any executor (SYNC)
- `ctx.get_state(key)` — retrieve stored data (SYNC)

### Routing Dataclass

We'll use a lightweight dataclass as the routing payload between `parse_triage` and the switch.

In [4]:
@dataclass
class RoutingDecision:
    """Lightweight payload used by edge conditions for routing."""
    severity: str
    service: str

print("✅ RoutingDecision dataclass defined")

✅ RoutingDecision dataclass defined


### Ingestion Executor (provided)

This is the **start node** — receives the raw alert JSON and forwards it to the triage agent.

Note how it constructs an `AgentExecutorRequest` with a `Message` object — this is how you
feed input to an `AgentExecutor` node.

In [5]:
@executor(id="ingest_alert")
async def ingest_alert(alert_json: str, ctx: WorkflowContext) -> None:
    """Start node: parse the alert and store it in workflow state, then forward to triage."""
    alert = json.loads(alert_json)
    
    # Store the alert in workflow state — accessible by all downstream executors
    ctx.set_state("alert", alert)
    ctx.set_state("service", alert["service"])
    
    # Forward to the triage agent as a user message
    user_msg = Message("user", contents=[
        f"New alert fired:\n"
        f"Title: {alert['title']}\n"
        f"Service: {alert['service']}\n"
        f"Type: {alert['incident_type']}\n"
        f"Description: {alert['description']}"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))

print("✅ ingest_alert executor defined")

✅ ingest_alert executor defined


### Parse Triage Result (provided)

This executor sits **between** the triage agent and the switch-case routing.
It parses the structured JSON output, stores the full result in state, and emits
a lightweight `RoutingDecision` for the edge conditions to inspect.

In [6]:
@executor(id="parse_triage")
async def parse_triage(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    """Parse triage agent's structured output and emit routing decision."""
    triage = TriageResult.model_validate_json(response.agent_response.text)
    
    # Store full triage result in workflow state for downstream use
    ctx.set_state("triage_result", triage)
    
    # Emit a lightweight routing payload for the switch-case edges
    await ctx.send_message(RoutingDecision(severity=triage.severity, service=ctx.get_state("service")))

print("✅ parse_triage executor defined")

✅ parse_triage executor defined


---
## Step 4: Edge Condition Functions

This is the **key MAF feature** — deterministic routing based on typed data.

You need condition functions that inspect the `RoutingDecision` payload and return `True`/`False`.

**Important:** You cannot have two `Case(target=X)` from the same source pointing to the same target
(duplicate edge error). Instead, combine conditions:

```python
# ✅ CORRECT: one condition, one target
Case(condition=lambda msg: msg.severity in ("critical", "high"), target=to_diagnostics)

# ❌ WRONG: two Cases pointing to same target = duplicate edge error
Case(condition=is_critical, target=to_diagnostics),
Case(condition=is_high, target=to_diagnostics),
```

In [7]:
def needs_diagnostics(msg: Any) -> bool:
    """Returns True for critical/high severity routing decisions."""
    return isinstance(msg, RoutingDecision) and msg.severity in ("critical", "high")

print(f"✅ needs_diagnostics defined")
print(f"   Test critical: {needs_diagnostics(RoutingDecision('critical','svc'))}")
print(f"   Test high: {needs_diagnostics(RoutingDecision('high','svc'))}")
print(f"   Test low: {needs_diagnostics(RoutingDecision('low','svc'))}")


✅ needs_diagnostics defined
   Test critical: True
   Test high: True
   Test low: False


---
## Step 5: Build the Branch Executors

### To-Diagnostics Executor

This executor handles the CRITICAL/HIGH path. It reads triage context from state
and creates an `AgentExecutorRequest` to send to the diagnostics agent.

In [8]:
@executor(id="to_diagnostics")
async def to_diagnostics(routing: RoutingDecision, ctx: WorkflowContext) -> None:
    """Critical/High path: prepare request for diagnostics agent."""
    triage: TriageResult = ctx.get_state("triage_result")
    service = ctx.get_state("service")
    user_msg = Message("user", contents=[
        f"Investigate this incident:\n"
        f"Service: {service}\n"
        f"Triage hypothesis: {triage.root_cause_hypothesis}\n"
        f"Investigate: {triage.recommended_action}"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))

print("✅ to_diagnostics executor defined")


✅ to_diagnostics executor defined


### Monitor-Only Executor

For LOW severity, we skip diagnostics/remediation entirely and just log the incident.
This is a **terminal node** — it uses `ctx.yield_output()` to produce the workflow result.

In [9]:
@executor(id="monitor_only")
async def monitor_only(routing: RoutingDecision, ctx: WorkflowContext) -> None:
    """Low-severity terminal: log and yield output without remediation."""
    alert = ctx.get_state("alert")
    await ctx.yield_output(
        f"📋 LOW severity: {alert['title']} — monitoring only, no action taken."
    )

print("✅ monitor_only executor defined")


✅ monitor_only executor defined


### Communications Executor (provided)

Terminal node for the critical/high path — formats the diagnostics result into a report.

In [10]:
@executor(id="comms")
async def comms(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    """Terminal: format diagnostics results into a workflow output report."""
    diag = DiagnosticsResult.model_validate_json(response.agent_response.text)
    service = ctx.get_state("service")
    triage: TriageResult = ctx.get_state("triage_result")
    
    ctx.set_state("diagnostics_result", diag)
    
    report = (
        f"\U0001f6a8 INCIDENT RESPONSE REPORT\n"
        f"{'='*40}\n"
        f"Service: {service}\n"
        f"Severity: {triage.severity.upper()}\n"
        f"Root Cause: {diag.root_cause}\n"
        f"Confidence: {diag.confidence:.0%}\n"
        f"Affected: {', '.join(diag.affected_components)}\n"
        f"Recommended Fix: {diag.recommended_fix}\n"
        f"Requires Restart: {diag.requires_restart}\n"
    )
    await ctx.yield_output(report)

print("✅ comms executor defined")

✅ comms executor defined


---
## Step 6: Wire the Workflow Graph

This is where it all comes together. Use `WorkflowBuilder` to create the DAG:

```
ingest_alert → triage_agent_exec → parse_triage → SWITCH:
    Case(needs_diagnostics) → to_diagnostics → diagnostics_agent_exec → comms
    Default                 → monitor_only
```

### API Reference:
```python
# Wrap Agent as a workflow node:
triage_agent_exec = AgentExecutor(triage_agent)

# Build the graph:
workflow = (
    WorkflowBuilder(start_executor=start_node)
    .add_edge(source, target)                          # Unconditional edge
    .add_switch_case_edge_group(source=parse_triage, cases=[
        Case(condition=needs_diagnostics, target=to_diagnostics),
        Default(target=monitor_only),
    ])
    .build()
)
```

In [11]:
triage_agent_exec = AgentExecutor(triage_agent)
diag_agent_exec = AgentExecutor(diagnostics_agent)

workflow = (
    WorkflowBuilder(start_executor=ingest_alert)
    .add_edge(ingest_alert, triage_agent_exec)
    .add_edge(triage_agent_exec, parse_triage)
    .add_switch_case_edge_group(parse_triage, [
        Case(condition=needs_diagnostics, target=to_diagnostics),
        Default(target=monitor_only),
    ])
    .add_edge(to_diagnostics, diag_agent_exec)
    .add_edge(diag_agent_exec, comms)
    .build()
)

print("✅ Workflow graph wired!")
print("   ingest_alert → triage → parse_triage → SWITCH")
print("     Case(critical/high) → to_diagnostics → diagnostics → comms")
print("     Default(low)        → monitor_only")


C:\Users\kiranpanchal\AppData\Local\Temp\ipykernel_11892\623464605.py:14: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()
Executor 'ingest_alert' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Executor 'parse_triage' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Executor 'parse_triage' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowConte

C:\Users\kiranpanchal\AppData\Local\Temp\ipykernel_11892\623464605.py:14: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()
Executor 'ingest_alert' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Executor 'parse_triage' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Executor 'parse_triage' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowConte

✅ Workflow graph wired!
   ingest_alert → triage → parse_triage → SWITCH
     Case(critical/high) → to_diagnostics → diagnostics → comms
     Default(low)        → monitor_only


---
## Step 7: Run the Workflow

Feed a CRITICAL incident and watch it route through diagnostics.
Then feed a LOW-severity incident and watch it take the monitor-only path.

In [12]:
# Run with CRITICAL incident (payment-api OOM)
print("\U0001f680 Running workflow with CRITICAL incident...")
print(f"   Alert: {incidents[0]['title']}\n")

result = await workflow.run(json.dumps(incidents[0]))
outputs = result.get_outputs()

if outputs:
    print(outputs[0])
else:
    print("❌ No output — check your edge conditions and executor logic")

🚀 Running workflow with CRITICAL incident...
   Alert: Payment API P99 Latency > 30s

{"severity":"high","is_recurring":true,"auto_remediation_allowed":true,"root_cause_hypothesis":"Recurring memory leaks in the Payment API's connection pool have led to an OOM issue, exacerbating latency spikes.","recommended_action":"Follow the High Latency Response playbook (RB-204): Restart the affected pod and clear the connection pool cache. Monitor P99 latency recovery and connection pool metrics. Investigate the batch job correlation for potential root cause fixes.","escalation_threshold_minutes":15}


In [14]:
# ✅ Validate the critical path was taken
assert outputs, "Workflow must produce output"
assert "Root Cause" in outputs[0], "Should include root cause from diagnostics"
print("✅ Critical path validation passed")

✅ Critical path validation passed
   Output type: AgentResponse
   Output preview: {"severity":"high","is_recurring":true,"auto_remediation_allowed":true,"root_cause_hypothesis":"Recurring memory leaks in the Payment API's connection...


In [15]:
# Run with LOW incident (notification-service rate limiting)
print("\U0001f680 Running workflow with LOW-severity incident...")
print(f"   Alert: {incidents[2]['title']}\n")

result_low = await workflow.run(json.dumps(incidents[2]))
outputs_low = result_low.get_outputs()

if outputs_low:
    print(outputs_low[0])
else:
    print("❌ No output — check the Default path")

🚀 Running workflow with LOW-severity incident...
   Alert: Notification Service Email Delivery Failing

{"severity":"medium","is_recurring":false,"auto_remediation_allowed":false,"root_cause_hypothesis":"Notification service is being rate-limited by SendGrid due to high email volume.","recommended_action":"Follow the Rate Limiting Response playbook (RB-410): Investigate quota usage and consider toggling the backend to a fallback provider temporarily. Notify the vendor relations team to negotiate rate-limits, if necessary.","escalation_threshold_minutes":20}


In [16]:
# ✅ Validate the low-severity path was taken
assert outputs_low, "Low-severity should still produce output"
assert "monitor" in outputs_low[0].lower() or "LOW" in outputs_low[0], \
    "Low-severity should take the monitor-only path"
print("✅ Low-severity routing validation passed")
print("\n\U0001f3af The SAME workflow handles both incidents differently based on severity!")

✅ Low-severity routing validation passed
   Output preview: {"severity":"medium","is_recurring":false,"auto_remediation_allowed":false,"root_cause_hypothesis":"Notification service is being rate-limited by Send

🎯 The SAME workflow handles both incidents differently based on severity!


---
## ➡️ Next: Challenge 3 — Human-in-the-Loop & Resilience

Your workflow routes correctly but executes autonomously.
In production, you need **human approval** before destructive actions
and **retry logic** when verification fails.

Challenge 3 adds:
- `ctx.request_info()` to pause the workflow and wait for human approval
- `@tool(approval_mode="always_require")` for dangerous operations
- Verification loop with retry-or-escalate logic

[Open Challenge 3 →](../challenge-3/challenge-3.ipynb)